In [1]:
import os
import sys
import time
import subprocess
import requests
import webbrowser
from pathlib import Path


In [2]:
os.environ["REDIS_URL"] = "redis://localhost:6379/0"

In [3]:
def start_infrastructure():
    print("Starting infrastructure services (postgres, redis)...")

    # Use compose.yml (docker compose)
    services = ["postgres", "redis"]

    try:
        result = subprocess.run(
            ["docker", "compose", "up", "-d", *services],
            capture_output=True,
            text=True,
            check=True,
        )
        print("Infra started ✅")
        if result.stdout:
            print(result.stdout.strip())
        time.sleep(3)
        return True
    except subprocess.CalledProcessError as e:
        print("Infra start failed ❌")
        print(e.stderr)
        return False

infra_ok = start_infrastructure()
infra_ok

Starting infrastructure services (postgres, redis)...
Infra started ✅


True

In [4]:
def show_compose_ps():
    # os.chdir(project_root)
    result = subprocess.run(["docker", "compose", "ps"], capture_output=True, text=True)
    print(result.stdout)

show_compose_ps()

NAME            IMAGE                COMMAND                   SERVICE         CREATED          STATUS                    PORTS
celery-worker   python:3.11-slim     "sh -c '\n  apt-get uâ€¦"   celery-worker   4 days ago       Up 42 seconds (healthy)   
postgres        postgres:15-alpine   "docker-entrypoint.sâ€¦"    postgres        40 minutes ago   Up 40 minutes (healthy)   0.0.0.0:5432->5432/tcp
redis           redis:7-alpine       "docker-entrypoint.sâ€¦"    redis           7 days ago       Up 7 days (healthy)       0.0.0.0:6379->6379/tcp



In [5]:
from src.config import AppConfig

cfg = AppConfig()

print("API host:", cfg.api.host)
print("API port:", cfg.api.port)
print("Postgres:", cfg.database.host, cfg.database.port, cfg.database.name)
print("Redis broker:", cfg.redis.broker_url)
print("Blob container:", cfg.azure.storage.container_name)
print("ACU endpoint:", cfg.acu.endpoint)
print("ACU analyzer_id:", cfg.acu.analyzer_id)

API host: 127.0.0.1
API port: 8000
Postgres: localhost 5432 credit_ocr
Redis broker: redis://localhost:6379/0
Blob container: documents
ACU endpoint: https://azure-foundry-westus-resource.services.ai.azure.com/
ACU analyzer_id: license_agreement_extraction_wrt_CUAD_v4_raw_normalized_singlepass


In [6]:
# run in a notebook cell before start_api_service()
import os, signal



In [7]:
api_process = None

API_HOST = cfg.api.host
API_PORT = int(cfg.api.port)
API_BASE_URL = f"http://{API_HOST}:{API_PORT}"

if 'api_process' in globals() and api_process is not None and api_process.poll() is None:
    api_process.terminate()
    try:
        api_process.wait(timeout=5)
    except Exception:
        api_process.kill()
api_process = None


def start_api_service():
    global api_process
    if api_process is not None:
        print("API already running")
        return True

    print("Starting FastAPI service...")
    # os.chdir(project_root)

    api_process = subprocess.Popen(
        [sys.executable, "run_api.py"],
        stdout=None,   # instead of subprocess.PIPE
        stderr=None,   # instead of subprocess.PIPE
        text=True,
    )

    print("API PID:", api_process.pid)

    # Wait for health check
    for attempt in range(20):
        try:
            r = requests.get(f"{API_BASE_URL}/api/v1/health", timeout=3)
            if r.status_code == 200:
                data = r.json()
                if data.get("status") in ("healthy", "ok"):
                    print("API healthy ✅")
                    return True
                else:
                    print("API responded, status:", data)
        except Exception as e:
            print(e)
            pass

        time.sleep(1.5)
        print(f"Waiting for API... ({attempt+1}/20)")

    print("API did not become healthy ❌")
    return False

api_ok = start_api_service()
api_ok


Starting FastAPI service...
API PID: 48148
API healthy ✅


True

In [8]:
print("Web UI:", f"{API_BASE_URL}/")
print("Docs:", f"{API_BASE_URL}/docs")
print("Health:", f"{API_BASE_URL}/api/v1/health")

try:
    webbrowser.open(f"{API_BASE_URL}/")
except Exception as e:
    print("Could not auto-open browser:", e)


Web UI: http://127.0.0.1:8000/
Docs: http://127.0.0.1:8000/docs
Health: http://127.0.0.1:8000/api/v1/health


In [9]:
from pathlib import Path
current_directory = Path.cwd()
pdf_path = current_directory / "data" / "AlliedEsportsEntertainmentInc_20190815_8-K_EX-10.19_11788293_EX-10.19_Content License Agreement.pdf"
assert pdf_path.exists(), f"Missing PDF: {pdf_path}"

files = {"file": (pdf_path.name, pdf_path.read_bytes(), "application/pdf")}
data = {"document_type": "license-agreement"}  # adjust if your API uses a different field name

resp = requests.post(f"{API_BASE_URL}/api/v1/upload", files=files, data=data, timeout=600)
print(resp.status_code)
print(resp.text)

upload_json = resp.json()
upload_json

200
{"document_id":"b50d20b4-0dc1-40d5-b8e4-585a057dd5f5","source_filename":"AlliedEsportsEntertainmentInc_20190815_8-K_EX-10.19_11788293_EX-10.19_Content License Agreement.pdf","document_type":"license-agreement","status":{"document_id":"b50d20b4-0dc1-40d5-b8e4-585a057dd5f5","text_extraction_status":"ready","processing_status":"pending extraction","extraction_jobs":[{"id":"0e5c22fc-9924-4fc9-8294-5dba58de563b","document_id":"b50d20b4-0dc1-40d5-b8e4-585a057dd5f5","created_at":"2026-02-17T06:17:42.796587Z","completed_at":null,"status":"pending","error_message":null}],"acu_result_blob_path":null}}


{'document_id': 'b50d20b4-0dc1-40d5-b8e4-585a057dd5f5',
 'source_filename': 'AlliedEsportsEntertainmentInc_20190815_8-K_EX-10.19_11788293_EX-10.19_Content License Agreement.pdf',
 'document_type': 'license-agreement',
 'status': {'document_id': 'b50d20b4-0dc1-40d5-b8e4-585a057dd5f5',
  'text_extraction_status': 'ready',
  'processing_status': 'pending extraction',
  'extraction_jobs': [{'id': '0e5c22fc-9924-4fc9-8294-5dba58de563b',
    'document_id': 'b50d20b4-0dc1-40d5-b8e4-585a057dd5f5',
    'created_at': '2026-02-17T06:17:42.796587Z',
    'completed_at': None,
    'status': 'pending',
    'error_message': None}],
  'acu_result_blob_path': None}}

In [10]:
doc_id = upload_json["document_id"]
resp = requests.post(f"{API_BASE_URL}/api/v1/documents/{doc_id}/trigger", timeout=300)
print(resp.status_code, resp.text)

200 {"document_id":"b50d20b4-0dc1-40d5-b8e4-585a057dd5f5","task_id":"d374d625-cc39-4867-9dfa-3972e7d99904","status":"queued"}


In [11]:
import time
for i in range(30):
    r = requests.get(f"{API_BASE_URL}/api/v1/documents/{doc_id}/status", timeout=30)
    s = r.json()
    print(i, s.get("processing_status"), s.get("extraction_jobs", [{}])[0].get("status"))
    if s.get("processing_status") in ("done", "failed"):
        break
    time.sleep(5)

0 pending extraction pending
1 acu running running
2 acu running running
3 acu running running
4 acu running running
5 acu running running
6 acu running running
7 acu running running
8 acu running running
9 acu running running
10 acu running running
11 acu running running
12 acu running running
13 done done


In [12]:
import psycopg2
from psycopg2.extras import RealDictCursor

doc_id = "b50d20b4-0dc1-40d5-b8e4-585a057dd5f5"  # paste document_id here

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    dbname="dms_meta",
    user="dms",
    password="dms",
)

with conn, conn.cursor(cursor_factory=RealDictCursor) as cur:
    cur.execute(
        """
        SELECT id, status, created_at, started_at, completed_at, error_message
        FROM extraction_jobs
        WHERE document_id = %s
        ORDER BY created_at DESC
        """,
        (doc_id,),
    )
    rows = cur.fetchall()

for r in rows:
    print(dict(r))


{'id': 'bbe74a64-dfd0-416c-9fba-51caf5cfbc65', 'status': 'done', 'created_at': datetime.datetime(2026, 2, 17, 6, 17, 43, 46555, tzinfo=datetime.timezone.utc), 'started_at': datetime.datetime(2026, 2, 17, 6, 17, 44, 897345, tzinfo=datetime.timezone.utc), 'completed_at': datetime.datetime(2026, 2, 17, 6, 18, 46, 325580, tzinfo=datetime.timezone.utc), 'error_message': None}
{'id': '0e5c22fc-9924-4fc9-8294-5dba58de563b', 'status': 'pending', 'created_at': datetime.datetime(2026, 2, 17, 6, 17, 42, 796587, tzinfo=datetime.timezone.utc), 'started_at': None, 'completed_at': None, 'error_message': None}


In [13]:
with conn, conn.cursor(cursor_factory=RealDictCursor) as cur:
    cur.execute(
        """
        SELECT id, text_extraction_status, processing_status, acu_result_blob_path, updated_at
        FROM documents
        WHERE id = %s
        """,
        (doc_id,),
    )
    print(dict(cur.fetchone() or {}))


{'id': 'b50d20b4-0dc1-40d5-b8e4-585a057dd5f5', 'text_extraction_status': 'completed', 'processing_status': 'done', 'acu_result_blob_path': 'acu/license-agreement/b50d20b4-0dc1-40d5-b8e4-585a057dd5f5.json', 'updated_at': datetime.datetime(2026, 2, 17, 6, 18, 46, 321250, tzinfo=datetime.timezone.utc)}
